# Competition notebook: Qwen two-stage LoRA

# Qwen-only reproduction of the proven two-stage LoRA pipeline

This version keeps the public-pool n-gram model, Qwen3-8B and Qwen3-14B two-stage LoRA checkpoints, centered twin, label calibration, sibling specialists, and balancing. OLMo2 and SmolLM3 components are intentionally excluded for an isolated first run.

# Two-stage LoRA on the public pool (LB 0.592)

**The problem in one line.** Telling Claude from GPT is easy. Telling `gemini 3.7 flash` from `gemini 3.8 flash` is the whole game, and the host gives us only 150 passages per model to learn it from.

**The idea.** Other Kagglers have published about 35,000 passages from these same 18 models, written to prompts in the competition's style. That is 13 times the training set. So we teach in two stages, the way you'd learn a language:

1. **Read a lot first.** Train a LoRA classifier on the public passages only. It learns what each model *sounds like*.
2. **Then listen to the teacher.** Fine tune it for 2 epochs on the host's 2,699 passages. It learns how each model sounded *when the host asked*: same settings, same formatting.

Stage 1 alone already matches a LoRA trained on the host data. Stage 2 adds another 9 points.

| Model | Local CV (5-fold, grouped by prompt) | Public LB |
|---|---|---|
| n-gram SVC on train + pool | 0.435 | |
| Qwen3-8B LoRA, public passages only (stage 1) | 0.483 | |
| Qwen3-8B LoRA, stage 1 + stage 2 | 0.569 | |
| Qwen3-14B LoRA, stage 1 + stage 2 | 0.569 | |
| **This notebook:** 8B + 14B + n-gram, centred twin, balancing | **0.593** | **0.592** |

The two LoRAs score the same alone but disagree on 22% of passages, which is why blending them helps.

**What runs here** (inference only, about 1 h on 2x T4 for the full test):
1. An n-gram model, fit here on train + pool.
2. Two LoRA checkpoints (Qwen3-8B and Qwen3-14B), trained offline. The training recipe is at the end.
3. A "centred twin" that compares passages answering the same question.
4. A log-probability blend, then class balancing.


In [ ]:
import gc, glob, json, os, re, time
import numpy as np
import pandas as pd
import torch
from torch import nn

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# Blend weights, fit on out-of-fold predictions (log probabilities are multiplied by these and summed).
W_SVC = 1.021
LORAS = {  # checkpoint folder: blend weight, centred-twin weight, batch size, base model to look for
    "lora8b_v1":  dict(w=0.226, twin_w=0.079, bs=16, base=dict(model_type="qwen3", hidden_size=4096, num_hidden_layers=36)),
    "lora14b_v1": dict(w=0.429, twin_w=0.150, bs=8,  base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
}
TWIN = dict(thresh=0.85, min_size=2, max_size=18)   # retained for diagnostics; contribution disabled below

# Column order of the checkpoints' heads (checked against each checkpoint's meta.json below).
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)
LABEL2I = {l: i for i, l in enumerate(LABELS)}


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
LOGIT = np.zeros((len(test), K))   # every member adds w * log p here
print("train", train.shape, "test", test.shape, flush=True)

## 1. The n gram model

Each model has typing habits. In the training data claude and GLM never use curly quotes, while gpt uses a curly apostrophe in about half its passages. Gemini almost never uses an em dash, while Claude uses one in three passages out of four. Character n grams pick this up.

We look at each passage four ways: raw characters +  words plus punctuation +  the *shape* of the text (`Xxxx xx x`) + function words only, with every content word replaced by `CONTENT` so the topic can't leak in. A linear SVM sits on top, trained on train + the public pool with balanced class weights.

**Change in this version.** The vectorisers are fitted once on the whole corpus and the fold loop in section 5 only slices rows, so an out-of-fold fit now costs one classifier instead of a full re-vectorisation. Row bookkeeping (`TR_ROWS`, `POOL_ROWS`, `EX_ROWS`, `Y_ALL`, `W_ALL`) is shared by every label-fitted member, which is what makes the out-of-fold harness in section 5 describe the model that is actually submitted.

In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
FUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]
def shape_text(t):
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)
def function_text(t):
    return " ".join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\w+|[^\w\s]", t))
pool = pd.read_parquet(find("pool_v1.parquet"))
pool = pool.dropna(subset=["text", "model"]).copy()
pool["model"] = pool["model"].astype(str)
pool["text"] = pool["text"].astype(str)
_unlabelled = int((~pool["model"].isin(LABELS)).sum())
if _unlabelled:
    print(f"dropping {_unlabelled} pool rows with a label outside the 18 classes", flush=True)
    pool = pool[pool["model"].isin(LABELS)].copy()

# Optional external augmentation.  The notebook remains runnable without it.
# Keep only in-style, competition-valid passages and give them a conservative weight.
extra_hits = sorted(glob.glob(f"{INPUT_DIR}/**/passages_clean.csv", recursive=True))
extra = pd.DataFrame(columns=["text", "model"])
if extra_hits:
    extra = pd.read_csv(extra_hits[0], usecols=lambda c: c in {"text", "model"})
    extra = extra.dropna(subset=["text", "model"]).copy()
    extra["model"] = extra["model"].astype(str)
    extra = extra[extra["model"].isin(LABELS)].copy()
    extra = extra[~extra["model"].isin({"kimi-k3", "gpt-5.6-luna"})].copy()
    extra["text"] = extra["text"].astype(str)
    word_count = extra["text"].str.split().str.len()
    extra = extra[word_count.between(40, 250)].copy()
    known_text = set(train["text"].astype(str)) | set(pool["text"].astype(str))
    extra = extra[~extra["text"].isin(known_text)].drop_duplicates("text")

train_texts = train.text.astype(str).tolist()
test_texts = test.text.astype(str).tolist()
pool_texts = pool.text.astype(str).tolist()
extra_texts = extra["text"].tolist()
fit_texts = train_texts + pool_texts + extra_texts

# Row bookkeeping: every label-fitted member (SVC, sibling heads, family heads,
# one-vs-rest heads) shares these index ranges, so the out-of-fold harness in
# section 5 can slice exactly the rows production uses.
POOL_W = 0.50
N_TR, N_POOL, N_EX = len(train), len(pool), len(extra)
TR_ROWS = np.arange(N_TR)
POOL_ROWS = np.arange(N_TR, N_TR + N_POOL)
EX_ROWS = np.arange(N_TR + N_POOL, N_TR + N_POOL + N_EX)
Y_ALL = np.r_[y,
              pool.model.map(LABEL2I).values.astype(int),
              extra["model"].map(LABEL2I).astype(int).values if N_EX else np.zeros(0, dtype=int)]
W_ALL = np.r_[np.ones(N_TR, dtype=np.float32),
              np.full(N_POOL, POOL_W, dtype=np.float32),
              np.full(N_EX, 0.35, dtype=np.float32)]

# The vectorisers are unsupervised, so they are fitted once on the whole corpus and
# the fold loop below only slices rows.  That keeps the OOF harness cheap.
A, B = [], []
for view, lo, hi, max_features, weight in VIEWS:
    prep = shape_text if view == "shape" else function_text if view == "function" else (lambda s: s)
    analyzer = "char" if view == "shape" else "word" if view == "function" else view
    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                max_features=max_features, dtype=np.float32)
    if analyzer == "word":
        opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
    vec = TfidfVectorizer(**opts)
    A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
    B.append(vec.transform([prep(t) for t in test_texts]) * weight)
X_ALL = hstack(A, format="csr")
X_TEST = hstack(B, format="csr")
del A, B
gc.collect()

# Dedicated character view for the sibling / family / one-vs-rest specialists.
pair_vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2,
                           max_features=160000, sublinear_tf=True, lowercase=False,
                           dtype=np.float32)
PAIR = pair_vec.fit_transform(fit_texts)
PAIR_TEST = pair_vec.transform(test_texts)

svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(
    X_ALL, Y_ALL, sample_weight=W_ALL)
P_SVC_TEST = softmax(svc.decision_function(X_TEST))
print(f"SVC rows: train={N_TR:,}, pool={N_POOL:,}, clean_extra={N_EX:,}  features={X_ALL.shape[1]:,}", flush=True)
del svc
gc.collect()
print(f"n-gram model done ({time.time() - T0:.0f}s)", flush=True)

## 2. The LoRA classifiers

Each checkpoint is a LoRA adapter on the **first 24 layers of Qwen3 8B** (or the first 27 of Qwen3 14B). We don't need the rest: style lives in the early and middle layers, while the last layers mostly encode the topic, which is useless here because test prompts are new.

For each passage we:
1. merge the adapter into the weights once (`W <- W + B A * alpha / r`), so no `peft` is needed
2. run the passage and take the mean hidden state at 8 depths (layers 2, 4, 6, 8, 12, 16, 20, 24), each through its own LayerNorm;
3. concatenate them (8 x 4096 numbers) and apply a linear layer to get 18 scores.

The final RMSNorm is switched off (`Identity`) because the checkpoints were trained that way. On a T4 everything runs in fp16, a batch that overflows is redone in bf16.


In [3]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


def find_base(spec):
    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f"attach the base model {spec}")


def load_trunk(base_dir, keep, dtype):
    """First `keep` decoder layers, split over both T4s when it does not fit on one."""
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, "layer_types", None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    hd = getattr(mc, "head_dim", mc.hidden_size // mc.num_attention_heads)
    gib = 2 * (mc.vocab_size * mc.hidden_size + keep * (2 * mc.hidden_size * mc.num_attention_heads * hd
               + 2 * mc.hidden_size * mc.num_key_value_heads * hd + 3 * mc.hidden_size * mc.intermediate_size)) / 2 ** 30
    if gib > 13 and torch.cuda.device_count() > 1:
        mem = {i: "13GiB" for i in range(torch.cuda.device_count())}
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
    else:
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
    model.norm = nn.Identity()
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir):
    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
    scale = acfg["lora_alpha"] / acfg["r"]
    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
    modules = dict(model.named_modules())
    for k, A in sd.items():
        if ".lora_A." in k:
            name = k.split(".lora_A.")[0]
            lin = modules[name[name.index("layers."):]]
            Bm = sd[k.replace(".lora_A.", ".lora_B.")]
            lin.weight.add_(((Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale).to(lin.weight.dtype))


@torch.no_grad()
def lora_features(ckpt_dir, base_dir, texts, bs):
    """Pooled features Z (n, 8 * hidden) and the checkpoint's heads."""
    meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
    cfg = meta["cfg"]
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = load_trunk(base_dir, cfg["keep"], dtype)
    merge_lora(model, os.path.join(ckpt_dir, "adapter"))
    heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
    layers, H = cfg["layers"], model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]
    ids = [x + [tok.eos_token_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together = less padding
    Z = np.zeros((len(ids), H * len(layers)), np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow: redo this batch in bf16
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            model.to(dtype)
        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
        m[:, 0] = 0   # the first token is an attention sink; leave it out of the mean
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()
    del model
    torch.cuda.empty_cache()
    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}


## 3. Compare answers to the same question

Picture two test passages that answer the same prompt. Whatever they share, the topic, is noise for us. Subtract the pair's average and what's left is mostly *how* each one was written.

So we cluster the test passages by topic (content word TFIDF, average linkage). In each cluster of 2 to 18 passages, we subtract the cluster mean from every passage's LoRA features and read them with a second head, which was trained during fine-tuning on exactly such differences. It adds a little evidence on top of the plain head.

How much this helps depends on how many test passages share a prompt, which we don't know. In simulation it gives +0.5 points at 1-2 passages per prompt and more with larger groups. With no clusters it simply does nothing.


In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test_texts)
D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
np.fill_diagonal(D, 0)
TEST_GROUPS = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance") \
    if len(test_texts) >= 2 else np.ones(len(test_texts), dtype=int)
size = np.bincount(TEST_GROUPS)[TEST_GROUPS]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
del D, Xt
print(f"topic clusters: {int((size == 1).sum())} singletons, {int(twin_rows.sum())} passages in clusters of 2-18", flush=True)

# ---------------------------------------------------------------------------
# Honest prompt-grouped OOF for the label-fitted members.
# The previous version fitted the fold SVC on the host rows only, while
# production fits it on train + the 34k public pool with sample weights, so the
# OOF number described a different model than the one being submitted.  The fold
# fits below use exactly the production rows (fold train + pool + extra).
# ---------------------------------------------------------------------------
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, recall_score

groups_oof = train["prompt_id"].astype(str).values if "prompt_id" in train else np.arange(N_TR).astype(str)
FOLDS = 3
FOLD_SPLITS = list(GroupKFold(FOLDS).split(np.arange(N_TR), y, groups_oof))
svc_oof = np.zeros((N_TR, K), np.float32)
for fi, (itr, iva) in enumerate(FOLD_SPLITS, 1):
    rows = np.r_[TR_ROWS[itr], POOL_ROWS, EX_ROWS]
    fold_svc = LinearSVC(C=0.5, random_state=100 + fi, max_iter=5000, class_weight="balanced")
    fold_svc.fit(X_ALL[rows], Y_ALL[rows], sample_weight=W_ALL[rows])
    svc_oof[iva] = softmax(fold_svc.decision_function(X_ALL[TR_ROWS[iva]]))
    print(f"OOF SVC fold {fi}/{FOLDS} done ({time.time() - T0:.0f}s)", flush=True)
    del fold_svc
    gc.collect()

all_text = train_texts + test_texts
q_train, q_test = {}, {}
for ckpt_name, mc in LORAS.items():
    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
    Z_all, h_all = lora_features(ckpt, find_base(mc["base"]), all_text, mc["bs"])
    q_all = softmax(Z_all @ h_all["plain.weight"].T + h_all["plain.bias"])
    q_train[ckpt_name], q_test[ckpt_name] = q_all[:len(train)], q_all[len(train):]
    del Z_all, h_all
    gc.collect()

# Blend weights are NOT re-fitted here.  q_train comes from heads that were
# fine-tuned on these very train rows, so any weight search on it is in-sample
# and pulls the blend towards the LoRAs.  The reference weights below come from
# the public two-stage notebook's own out-of-fold fit, and section 5 chooses the
# post-processing switches on svc_oof (which is genuinely out-of-fold) instead.
ref_ba = balanced_accuracy_score(y, (W_SVC * np.log(np.clip(svc_oof, 1e-9, 1))
    + LORAS["lora8b_v1"]["w"] * np.log(np.clip(q_train["lora8b_v1"], 1e-9, 1))
    + LORAS["lora14b_v1"]["w"] * np.log(np.clip(q_train["lora14b_v1"], 1e-9, 1))).argmax(1))
print(f"frozen reference weights {W_SVC}/{LORAS['lora8b_v1']['w']}/{LORAS['lora14b_v1']['w']}, "
      f"mixed-stack OOF reference BA={ref_ba:.5f}", flush=True)
LOGIT += W_SVC * np.log(np.clip(P_SVC_TEST, 1e-9, 1))


# Optional Qwen3 embedding-SVM phase. The attachment is not required: when
# the embedding model is absent, the proven OOF blend runs unchanged.
embed_configs = sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True))
embed_hits = [os.path.dirname(x) for x in embed_configs
              if "qwen" in x.lower() and "embedding" in x.lower()]
if embed_hits:
    import torch.nn.functional as F
    embed_dir = embed_hits[0]
    print(f"embedding model found: {embed_dir}", flush=True)
    embed_tok = AutoTokenizer.from_pretrained(embed_dir, padding_side="left", local_files_only=True)
    embed_model = AutoModel.from_pretrained(embed_dir, torch_dtype=torch.float16,
                                             local_files_only=True).cuda().eval()
    def _embed(items, bs=8, max_len=512):
        out = []
        for start in range(0, len(items), bs):
            tok = embed_tok(items[start:start + bs], padding=True, truncation=True,
                            max_length=max_len, return_tensors="pt")
            tok = {k: v.cuda() for k, v in tok.items()}
            with torch.inference_mode():
                h = embed_model(**tok).last_hidden_state
                mask = tok["attention_mask"]
                left = bool((mask[:, -1].sum() == mask.shape[0]).item())
                if left:
                    pooled = h[:, -1]
                else:
                    pos = mask.sum(1) - 1
                    pooled = h[torch.arange(h.shape[0], device=h.device), pos]
                out.append(F.normalize(pooled.float(), p=2, dim=1).cpu().numpy())
        return np.vstack(out).astype(np.float32)
    emb_all = _embed(all_text)
    emb_train, emb_test = emb_all[:len(train)], emb_all[len(train):]
    emb_scores = {}
    for C_emb in (0.3, 1.0, 3.0):
        fold_scores = []
        for itr, iva in FOLD_SPLITS:
            em = LinearSVC(C=C_emb, class_weight="balanced", max_iter=3000, random_state=31)
            em.fit(emb_train[itr], y[itr])
            fold_scores.append(balanced_accuracy_score(y[iva], em.predict(emb_train[iva])))
        emb_scores[C_emb] = float(np.mean(fold_scores))
    best_C_emb = max(emb_scores, key=emb_scores.get)
    emb_model = LinearSVC(C=best_C_emb, class_weight="balanced", max_iter=5000, random_state=31)
    emb_model.fit(emb_train, y)
    P_EMB = softmax(emb_model.decision_function(emb_test))
    LOGIT += 0.18 * np.log(np.clip(P_EMB, 1e-9, 1))
    print(f"embedding-SVM grouped BA={emb_scores}, selected_C={best_C_emb}", flush=True)
    del embed_model, embed_tok, emb_all, emb_train, emb_test, emb_model
    torch.cuda.empty_cache(); gc.collect()
else:
    print("embedding model not attached; embedding-SVM phase skipped", flush=True)

for ckpt_name, mc in LORAS.items():
    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
    Z, heads = lora_features(ckpt, find_base(mc["base"]), test_texts, mc["bs"])
    LOGIT += mc["w"] * np.log(np.clip(softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), 1e-9, 1))
    if twin_rows.any():
        # Leave-one-out cavity centering: do not let each passage cancel
        # itself out of the topic mean. Shrink the larger n=2 correction so
        # its scale remains comparable to the original centred head.
        Zc = np.zeros_like(Z)
        for g in np.unique(TEST_GROUPS[twin_rows]):
            idx = np.flatnonzero(TEST_GROUPS == g)
            group_sum = Z[idx].sum(0)
            n = len(idx)
            for row in idx:
                Zc[row] = (Z[row] - (group_sum - Z[row]) / max(n - 1, 1)) * ((n - 1) / n)
        centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]
        LOGIT[twin_rows] += mc["twin_w"] * np.log(np.clip(softmax(centred), 1e-9, 1))
    del Z
    gc.collect()
    print(f"{ckpt_name} done ({time.time() - T0:.0f}s)", flush=True)

## 5. Post-processing chosen on out-of-fold evidence

Everything that learns from labels — sibling-pair heads, the family gate, the within-family heads, one-vs-rest heads for the weakest classes, the recall calibration — is now fitted on **train + the public pool** (34k rows) instead of the 2,699 host rows alone. The pool is exactly where siblings separate, and those are the classes that drag balanced accuracy down.

Every switch (pool specialists on/off, one-vs-rest heads, prompt centering strength, shield strength, marginal balancing strength, calibration) is then accepted or rejected on **prompt-grouped out-of-fold balanced accuracy**, with a 0.002 acceptance margin. The 3-row interactive test is only used to confirm the pipeline runs; it cannot show a score change, so no candidate is judged on it any more.

Two things are deliberately *not* re-fitted: the blend weights. `q_train` comes from heads fine-tuned on those same train rows, so a weight search on it is in-sample and pulls the blend towards the LoRAs. The reference weights from the public two-stage notebook stay frozen.

In [ ]:
# ===========================================================================
# 5. Evidence-driven post-processing
#    Every label-fitted specialist below is now trained on train + the public
#    pool (13x more rows, and the pool is exactly where siblings separate),
#    and every switch is chosen on prompt-grouped out-of-fold predictions
#    instead of on the 3-row interactive test, which cannot show anything.
# ===========================================================================
FAMILY_NAMES = ["OpenAI", "Google", "Anthropic", "Qwen", "ZAI", "Moonshot"]
def family_of(lab):
    for pre, fam in (("gpt-", "OpenAI"), ("gemini-", "Google"), ("claude-", "Anthropic"),
                     ("qwen", "Qwen"), ("glm-", "ZAI"), ("kimi-", "Moonshot")):
        if lab.startswith(pre):
            return fam
    return "Qwen"
FAMILY_OF = {l: family_of(l) for l in LABELS}
FAM_IDX = np.array([FAMILY_NAMES.index(FAMILY_OF[l]) for l in LABELS])
FAMILY_Y = FAM_IDX[Y_ALL]

PAIRS = [("gemini-3.6-flash", "gemini-3.7-flash"), ("gemini-3.7-flash", "gemini-3.8-flash"),
         ("gemini-3.1-pro-preview", "gemini-3.7-flash"), ("gemini-3.1-pro-preview", "gemini-3.8-flash"),
         ("gpt-5.6-luna", "gpt-5.6-sol"), ("gpt-5.6-luna", "gpt-5.6-terra"),
         ("gpt-5.6-sol", "gpt-5.6-terra"), ("gpt-5.5", "gpt-6-astra"),
         ("glm-5.3", "glm-5.3-flash"),
         ("claude-opus-4-6", "claude-opus-4-8"), ("claude-opus-4-8", "claude-opus-5"),
         ("claude-opus-4-6", "claude-opus-5"), ("claude-sonnet-5", "claude-opus-5"),
         ("claude-fable-5", "claude-sonnet-5")]
# Classes with the lowest out-of-fold recall: the metric is macro recall, so a
# single one of these recovering 0.20 of its passages is worth ~0.011 score.
WORST = ["claude-fable-5", "gpt-5.6-sol", "glm-5.3", "glm-5.3-flash", "gpt-5.6-luna",
         "gpt-5.6-terra", "gemini-3.8-flash", "gemini-3.7-flash"]

SPEC = dict(pair_gap=0.30, pair_delta=0.16, fam_gap=0.55, fam_p=0.60, fam_delta=0.08,
            hier_gap=0.55, hier_p=0.60, hier_delta=0.10, ovr_gap=0.30, ovr_delta=0.12)

OOF_RECALL = {
    "claude-fable-5": 0.0738, "gpt-5.6-sol": 0.1200,
    "glm-5.3": 0.1333, "glm-5.3-flash": 0.1400,
    "gpt-5.6-luna": 0.2533, "gpt-5.6-terra": 0.2600,
    "gemini-3.8-flash": 0.3000, "gemini-3.7-flash": 0.3400,
}
MEAN_RECALL = 0.3389
CLAUDE_GLM_IDX = [i for i, lab in enumerate(LABELS) if lab.startswith("claude-") or lab.startswith("glm-")]
GEMINI_IDX = [i for i, lab in enumerate(LABELS) if lab.startswith("gemini-")]
GPT_SOL_IDX = LABEL2I["gpt-5.6-sol"]
CURLY_RE = re.compile(r"[\u2018\u2019\u201c\u201d]")


def _probs(scores):
    """Row-wise softmax that also handles a binary head's 1-D decision_function."""
    scores = np.asarray(scores, dtype=np.float64)
    if scores.ndim == 1:
        scores = np.c_[-scores, scores]
    e = np.exp(scores - scores.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def _margins(L):
    P = softmax(L)
    top2 = np.argsort(L, 1)[:, -2:]
    return P, P.max(1), top2


def specialist_deltas(fit_rows, A_apply, L_apply, tag=""):
    """Delta logits from sibling-pair, family, hierarchical and one-vs-rest heads.

    `fit_rows` index PAIR/Y_ALL (fold train rows + the whole pool); `A_apply` is
    the matching feature block for the rows we are scoring.
    """
    D_pair = np.zeros_like(L_apply)
    D_fam = np.zeros_like(L_apply)
    D_hier = np.zeros_like(L_apply)
    D_ovr = np.zeros_like(L_apply)
    P, p1, top2 = _margins(L_apply)
    gap = p1 - np.sort(P, 1)[:, -2]

    # (a) sibling pairs, fitted on train + pool instead of train only
    n_pair = 0
    for left, right in PAIRS:
        il, ir = LABEL2I[left], LABEL2I[right]
        amb = np.flatnonzero(np.isin(top2, [il, ir]).all(1) & (gap < SPEC["pair_gap"]))
        rows = fit_rows[np.isin(Y_ALL[fit_rows], [il, ir])]
        if len(amb) == 0 or len(rows) < 40:
            continue
        m = LinearSVC(C=0.35, class_weight="balanced", max_iter=3000, random_state=17)
        m.fit(PAIR[rows], Y_ALL[rows])
        s = np.tanh(np.clip(m.decision_function(A_apply[amb]), -2.0, 2.0)) * SPEC["pair_delta"]
        sign = 1.0 if list(m.classes_)[1] == ir else -1.0
        D_pair[amb, ir] += sign * s
        D_pair[amb, il] -= sign * s
        n_pair += len(amb)

    # (b) family gate on cross-family ambiguity
    fam_m = LinearSVC(C=0.20, class_weight="balanced", max_iter=3000, random_state=23)
    fam_m.fit(PAIR[fit_rows], FAMILY_Y[fit_rows])
    fam_p = _probs(fam_m.decision_function(A_apply))
    main_fam = FAM_IDX[L_apply.argmax(1)]
    runner_fam = FAM_IDX[top2[:, 0]]
    cross = (main_fam != runner_fam) & (gap < SPEC["fam_gap"]) & (fam_p.max(1) > SPEC["fam_p"])
    if cross.any():
        rc = np.flatnonzero(cross)
        for fi in range(len(FAMILY_NAMES)):
            cols = np.flatnonzero(FAM_IDX == fi)
            D_fam[np.ix_(rc, cols)] += (SPEC["fam_delta"]
                                        * np.log(np.clip(fam_p[rc, fi], 1e-6, 1.0))[:, None])

    # (c) hierarchical family specialist for the sibling decision inside a family
    hier_rows = (fam_p.max(1) > SPEC["hier_p"]) & (gap < SPEC["hier_gap"])
    for fi, fam in enumerate(FAMILY_NAMES):
        rows_a = np.flatnonzero(hier_rows & (fam_p.argmax(1) == fi))
        rows_f = fit_rows[FAMILY_Y[fit_rows] == fi]
        labs = np.unique(Y_ALL[rows_f])
        if len(rows_a) == 0 or len(labs) < 2:
            continue
        m = LinearSVC(C=0.35, class_weight="balanced", max_iter=3000, random_state=41 + fi)
        m.fit(PAIR[rows_f], Y_ALL[rows_f])
        lp = np.log(np.clip(_probs(m.decision_function(A_apply[rows_a])), 1e-9, 1.0))
        for j, lab in enumerate(m.classes_):
            D_hier[rows_a, lab] += SPEC["hier_delta"] * lp[:, j]

    # (d) one-vs-rest heads for the classes the metric punishes most
    n_ovr = 0
    for lab in WORST:
        il = LABEL2I[lab]
        cand = (L_apply.argmax(1) == il) | np.isin(top2, [il]).any(1)
        amb = np.flatnonzero(cand & (gap < SPEC["ovr_gap"]))
        target = (Y_ALL[fit_rows] == il).astype(int)
        if len(amb) == 0 or target.sum() < 20 or target.sum() > len(target) - 20:
            continue
        # keep every positive, sample 3x as many negatives: same signal, far less runtime
        pos = np.flatnonzero(target == 1)
        neg = np.flatnonzero(target == 0)
        if len(neg) > 3 * len(pos):
            neg = np.random.default_rng(59 + il).choice(neg, 3 * len(pos), replace=False)
        sub_rows = np.sort(np.r_[pos, neg])
        m = LinearSVC(C=0.5, class_weight="balanced", max_iter=3000, random_state=59 + il)
        m.fit(PAIR[fit_rows[sub_rows]], target[sub_rows])
        s = np.tanh(np.clip(m.decision_function(A_apply[amb]), -2.0, 2.0)) * SPEC["ovr_delta"]
        D_ovr[amb, il] += s
        n_ovr += len(amb)

    print(f"  specialists{tag}: sibling rows={n_pair}, cross-family rows={int(cross.sum())}, "
          f"within-family rows={int(hier_rows.sum())}, one-vs-rest rows={n_ovr}", flush=True)
    return D_pair, D_fam, D_hier, D_ovr


def center_logits(L, groups_arr, lam, min_size=2, max_size=18):
    """Subtract each topic cluster's leave-one-out mean from its passages' logits."""
    if lam <= 0:
        return L
    out = L.copy()
    for g in np.unique(groups_arr):
        idx = np.flatnonzero(groups_arr == g)
        n = len(idx)
        if min_size <= n <= max_size:
            loo = (L[idx].sum(0) - L[idx]) / (n - 1)
            out[idx] = L[idx] - lam * loo
    return out


def postprocess(L, texts, groups_arr, cfg):
    Lc = center_logits(L, groups_arr, cfg["lam"])
    if cfg["calib"] > 0:
        for lab, rec in OOF_RECALL.items():
            Lc[:, LABEL2I[lab]] += float(np.clip(cfg["calib"] * np.log(MEAN_RECALL / rec), -0.12, 0.12))
    P = softmax(Lc)
    if cfg["alpha"] > 0 and len(P) >= 50:
        target = np.full((1, K), len(P) / K, dtype=np.float64)
        for _ in range(40):
            column_mass = P.sum(0, keepdims=True).clip(min=1e-12)
            P *= (target / column_mass) ** cfg["alpha"]
            P /= P.sum(1, keepdims=True).clip(min=1e-12)
    if cfg["shield"] != "off":
        f_curly, f_dash, f_apos = ((0.01, 0.05, 0.01) if cfg["shield"] == "hard"
                                   else (0.25, 0.50, 0.50))
        for i, txt in enumerate(texts):
            if CURLY_RE.search(txt):
                P[i, CLAUDE_GLM_IDX] *= f_curly
            if "\u2014" in txt:
                P[i, GEMINI_IDX] *= f_dash
            if "'" in txt:
                P[i, GPT_SOL_IDX] *= f_apos
    P /= P.sum(1, keepdims=True).clip(min=1e-12)
    return P


# --- out-of-fold deltas: fit on each fold's train rows + pool, score its holdout
D_OOF = {k: np.zeros((N_TR, K), np.float32) for k in ("pair", "fam", "hier", "ovr")}
for fi, (itr, iva) in enumerate(FOLD_SPLITS, 1):
    fit_rows = np.r_[TR_ROWS[itr], POOL_ROWS, EX_ROWS]
    val_rows = TR_ROWS[iva]
    d = specialist_deltas(fit_rows, PAIR[val_rows], W_SVC * np.log(np.clip(svc_oof[iva], 1e-9, 1)),
                          tag=f" fold {fi}")
    for k, v in zip(("pair", "fam", "hier", "ovr"), d):
        D_OOF[k][iva] = v
    del d
    gc.collect()

L_OOF = W_SVC * np.log(np.clip(svc_oof, 1e-9, 1))
D_SPEC_OOF = D_OOF["pair"] + D_OOF["fam"] + D_OOF["hier"]
D_OVR_OOF = D_OOF["ovr"]


def oof_score(cfg):
    L = L_OOF + (D_SPEC_OOF if cfg["spec"] else 0) + (D_OVR_OOF if cfg["ovr"] else 0)
    P = postprocess(L, train_texts, groups_oof, cfg)
    return balanced_accuracy_score(y, P.argmax(1))


BASE_CFG = dict(spec=False, ovr=False, lam=0.0, alpha=0.5, shield="hard", calib=0.10)
STEPS = [("pool-trained sibling/family specialists", dict(spec=True)),
         ("one-vs-rest worst-class heads", dict(ovr=True)),
         ("final-logit prompt centering 0.15", dict(lam=0.15)),
         ("final-logit prompt centering 0.30", dict(lam=0.30)),
         ("soft orthographic shield", dict(shield="soft")),
         ("shield off", dict(shield="off")),
         ("no marginal balancing", dict(alpha=0.0)),
         ("stronger marginal balancing", dict(alpha=0.75)),
         ("no recall calibration", dict(calib=0.0))]
best_cfg = dict(BASE_CFG)
best_ba = oof_score(best_cfg)
print(f"\nOOF stage 0 (v40 recipe, honest folds): BA={best_ba:.5f}", flush=True)
for name, patch in STEPS:
    cand = dict(best_cfg)
    cand.update(patch)
    ba = oof_score(cand)
    accepted = ba > best_ba + 0.002
    if accepted:
        best_cfg, best_ba = cand, ba
    print(f"OOF stage {name}: BA={ba:.5f} (best {best_ba:.5f}){'' if not accepted else '  <== accepted'}",
          flush=True)
print(f"OOF chosen config: {best_cfg}  BA={best_ba:.5f}\n", flush=True)

P_OOF_BEST = postprocess(L_OOF + (D_SPEC_OOF if best_cfg["spec"] else 0)
                         + (D_OVR_OOF if best_cfg["ovr"] else 0), train_texts, groups_oof, best_cfg)
P_OOF_BASE = postprocess(L_OOF, train_texts, groups_oof, BASE_CFG)
rec_base = recall_score(y, P_OOF_BASE.argmax(1), average=None, labels=np.arange(K))
rec_best = recall_score(y, P_OOF_BEST.argmax(1), average=None, labels=np.arange(K))
print("OOF per-class recall (v40 recipe -> chosen config):", flush=True)
for i, lab in enumerate(LABELS):
    print(f"  {lab:<24} {rec_base[i]:.3f} -> {rec_best[i]:.3f}", flush=True)

def tight_groups(texts, groups_arr, min_sim=0.35, min_size=2, max_size=18):
    """Keep a topic cluster for centering only when every pair inside it is similar.

    The agglomerative threshold is loose (distance 0.85 on 1 - cosine), and on the
    hidden test a wrongly merged cluster would subtract the wrong topic mean, so a
    single loose pair demotes the whole cluster to singletons.
    """
    out = groups_arr.copy()
    if len(texts) < 2:
        return out
    Xs = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(texts)
    S = (Xs @ Xs.T).toarray()
    kept = 0
    for g in np.unique(groups_arr):
        idx = np.flatnonzero(groups_arr == g)
        n = len(idx)
        ok = min_size <= n <= max_size
        if ok and n > 2:
            sub = S[np.ix_(idx, idx)]
            iu = np.triu_indices(n, 1)
            ok = bool(sub[iu].min() >= min_sim)
        if ok:
            kept += 1
        else:
            out[idx] = 10 ** 6 + g * 10 ** 3 + np.arange(n)
    print(f"  centering clusters kept {kept}/{len(np.unique(groups_arr))} "
          f"({int((out >= 10 ** 6).sum())} passages demoted to singletons)", flush=True)
    return out


# --- apply the chosen config to the real test stack -------------------------
FULL_FIT_ROWS = np.r_[TR_ROWS, POOL_ROWS, EX_ROWS]
d_test = specialist_deltas(FULL_FIT_ROWS, PAIR_TEST, LOGIT.copy(), tag=" test")
L_TEST = LOGIT + (d_test[0] + d_test[1] + d_test[2] if best_cfg["spec"] else 0) \
    + (d_test[3] if best_cfg["ovr"] else 0)
CENTRE_GROUPS = tight_groups(test_texts, TEST_GROUPS) if best_cfg["lam"] > 0 else TEST_GROUPS
P_TEST = postprocess(L_TEST, test_texts, CENTRE_GROUPS, best_cfg)
P_TEST_REF = postprocess(LOGIT, test_texts, CENTRE_GROUPS, BASE_CFG)
print(f"\ninteractive-test rows: {len(test_texts)}, "
      f"changed by the chosen config: {int((P_TEST.argmax(1) != P_TEST_REF.argmax(1)).sum())}", flush=True)

In [ ]:
sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P_TEST.argmax(1)]})
sub.to_parquet("submission.parquet", index=False)

report = {
    "chosen_config": {k: (v if isinstance(v, (bool, str)) else float(v)) for k, v in best_cfg.items()},
    "oof_ba_v40_recipe": float(oof_score(BASE_CFG)),
    "oof_ba_chosen": float(best_ba),
    "oof_per_class_recall_v40": {lab: float(rec_base[i]) for i, lab in enumerate(LABELS)},
    "oof_per_class_recall_chosen": {lab: float(rec_best[i]) for i, lab in enumerate(LABELS)},
    "blend_weights": {"svc": W_SVC, **{k: v["w"] for k, v in LORAS.items()}},
    "rows": {"train": int(N_TR), "pool": int(N_POOL), "extra": int(N_EX), "test": int(len(test_texts))},
}
with open("validation_results.json", "w") as f:
    json.dump(report, f, indent=2)
print(sub["model"].value_counts().to_string())
print(f"done in {time.time() - T0:.0f}s")